# BA工作坊 Task 2 - 技术指标计算与分析

**股票：芯动联科（688582.SH）**  
**数据区间：2025-07-04 ~ 2026-07-03**  
**分析工具：Python (pandas, numpy, matplotlib)**

## 一、环境配置与数据加载

In [ ]:
# 导入所需库
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings("ignore")

# 中文字体配置
font_paths = fm.findSystemFonts(fontpaths=None, fontext="ttf")
cn_font = None
for fp in font_paths:
    if any(k in fp.lower() for k in ["msyh", "simhei", "simsun", "microsoftyahei"]):
        cn_font = fp
        break
if cn_font:
    prop = fm.FontProperties(fname=cn_font)
    plt.rcParams["font.family"] = prop.get_name()
plt.rcParams["axes.unicode_minus"] = False

print("环境配置完成")

In [ ]:
# 加载已存储的股价数据
csv_path = r"C:\Users\TaoZi\Desktop\ED\BA工作坊\芯动联科_688582_年度日交易数据.csv"
df = pd.read_csv(csv_path, encoding="utf-8-sig")
df["trade_date"] = pd.to_datetime(df["trade_date"], format="%Y%m%d")
df = df.sort_values("trade_date").reset_index(drop=True)

print(f"数据加载完成，共 {len(df)} 条记录")
print(f"日期范围：{df['trade_date'].min().date()} ~ {df['trade_date'].max().date()}")
df.head()

## 二、数据基础诊断分析

在对数据进行技术指标计算之前，需要对数据的完整性和质量进行诊断分析，包括缺失值检查、描述性统计量计算等。

In [ ]:
# 缺失值检查
print("=== 缺失值检查 ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "无缺失值（所有字段完整）")

# 描述性统计量
print("\n=== 收盘价描述性统计 ===")
print(df["close"].describe())

print("\n=== 成交量描述性统计 ===")
print(df["vol"].describe())

## 三、RSI（相对强弱指标）

### 3.1 RSI的计算方法

RSI（Relative Strength Index）由J. Welles Wilder于1978年提出，通过计算一定时期内价格上涨和下跌幅度的比值，来衡量市场的超买超卖状态。

**计算步骤（以14日RSI为例）：**

1. 计算每日价格变化：`Δ = close[t] - close[t-1]`
2. 区分正收益和负收益：
   - gain = max(Δ, 0)（上涨幅度，下跌时为0）
   - loss = max(-Δ, 0)（下跌幅度，上涨时为0）
3. 计算平均收益和平均损失（使用指数移动平均EMA）：
   - avg_gain = EMA(gain, 14)
   - avg_loss = EMA(loss, 14)
4. 计算相对强度：RS = avg_gain / avg_loss
5. 计算RSI：RSI = 100 - 100 / (1 + RS)

**RSI的取值范围：** 0 ~ 100

### 3.2 RSI的作用

- **超买超卖判断**：RSI > 80 为超买区（价格可能回调）；RSI < 20 为超卖区（价格可能反弹）
- **背离信号**：价格创新高但RSI未创新高 → 顶背离（卖出信号）；价格创新低但RSI未创新低 → 底背离（买入信号）
- **中线突破**：RSI由下向上突破50 → 多头信号；RSI由上向下跌破50 → 空头信号

In [ ]:
def calc_rsi(close, period=14):
    """计算RSI指标"""
    delta = close.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.ewm(com=period-1, adjust=False).mean()
    avg_loss = loss.ewm(com=period-1, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

df["RSI_14"] = calc_rsi(df["close"])
print("RSI计算完成")
print(f"RSI最新值: {df['RSI_14'].iloc[-1]:.2f}")
df[["trade_date", "close", "RSI_14"]].tail(10)

## 四、MACD指标

### 4.1 MACD的计算方法

MACD（Moving Average Convergence Divergence）由Gerald Appel于1970年代提出，是基于指数移动平均线的趋势跟踪指标，由三条线组成。

**计算步骤（经典参数12, 26, 9）：**

1. 计算快线EMA：`EMA_fast = EMA(close, 12)`
2. 计算慢线EMA：`EMA_slow = EMA(close, 26)`
3. 计算MACD线：`MACD_line = EMA_fast - EMA_slow`
4. 计算信号线：`Signal_line = EMA(MACD_line, 9)`
5. 计算柱状图：`Histogram = MACD_line - Signal_line`

### 4.2 MACD的作用

- **趋势判断**：MACD线在零轴上方 → 多头市场；MACD线在零轴下方 → 空头市场
- **买卖信号**：MACD线由下向上穿越Signal线 → 金叉（买入）；MACD线由上向下穿越Signal线 → 死叉（卖出）
- **柱状图**：柱状图由负转正 → 上涨动能增强；由正转负 → 下跌动能增强
- **背离**：价格新高但MACD不创新高 → 顶背离（卖出信号）

In [ ]:
def calc_macd(close, fast=12, slow=26, signal=9):
    """计算MACD指标"""
    ema_fast = close.ewm(span=fast, adjust=False).mean()
    ema_slow = close.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram = macd_line - signal_line
    return macd_line, signal_line, histogram

macd_line, signal_line, histogram = calc_macd(df["close"])
df["MACD"] = macd_line
df["MACD_signal"] = signal_line
df["MACD_hist"] = histogram
print("MACD计算完成")
df[["trade_date", "close", "MACD", "MACD_signal", "MACD_hist"]].tail(10)

## 五、布林带（Bollinger Bands）

### 5.1 布林带的计算方法

布林带由John Bollinger于1980年代提出，利用统计学原理构建价格的波动通道，由三条轨道线组成。

**计算步骤（经典参数20, 2）：**

1. 计算中轨：`Mid = SMA(close, 20)`（20日简单移动平均）
2. 计算标准差：`std = std(close, 20)`
3. 计算上轨：`Upper = Mid + 2 × std`
4. 计算下轨：`Lower = Mid - 2 × std`

> 根据正态分布原理，约95%的价格数据应落在布林带（±2倍标准差）范围内。

### 5.2 布林带的作用

- **超买超卖**：价格触及上轨 → 可能回调；价格触及下轨 → 可能反弹
- **波动率判断**：布林带收口（带宽变窄）→ 低波动，预示即将变盘；布林带开口 → 高波动
- **趋势判断**：强势上涨中价格沿上轨运行；强势下跌中价格沿下轨运行
- **支撑阻力**：中轨（20日均线）是重要的动态支撑/阻力位

In [ ]:
def calc_bollinger(close, period=20, std_dev=2):
    """计算布林带指标"""
    mid = close.rolling(window=period).mean()
    std = close.rolling(window=period).std()
    upper = mid + std_dev * std
    lower = mid - std_dev * std
    return upper, mid, lower

bb_upper, bb_mid, bb_lower = calc_bollinger(df["close"])
df["BB_upper"] = bb_upper
df["BB_mid"] = bb_mid
df["BB_lower"] = bb_lower
print("布林带计算完成")
df[["trade_date", "close", "BB_upper", "BB_mid", "BB_lower"]].tail(10)

## 六、KDJ指标（随机指标）

### 6.1 KDJ的计算方法

KDJ指标由George Lane提出，是随机指标（Stochastic Oscillator）的扩展，通过考察收盘价在最近n日最高价和最低价之间的相对位置，判断超买超卖状态。

**计算步骤（经典参数9）：**

1. 计算n日最低价和最高价：
   - low_n = min(low, 9)
   - high_n = max(high, 9)
2. 计算RSV：`RSV = (close - low_n) / (high_n - low_n) × 100`
3. 计算K值：`K = EMA(RSV, 3)`（对RSV做平滑）
4. 计算D值：`D = EMA(K, 3)`（对K做平滑）
5. 计算J值：`J = 3 × K - 2 × D`

### 6.2 KDJ的作用

- **超买超卖**：K、D > 80 → 超买；K、D < 20 → 超卖
- **买卖信号**：K由下向上穿越D → 金叉（买入）；K由上向下穿越D → 死叉（卖出）
- **J值参考**：J > 100 → 极端超买；J < 0 → 极端超卖（J值最敏感）

In [ ]:
def calc_kdj(high, low, close, period=9):
    """计算KDJ指标"""
    low_n = low.rolling(window=period).min()
    high_n = high.rolling(window=period).max()
    rsv = (close - low_n) / (high_n - low_n) * 100
    k = rsv.ewm(com=2, adjust=False).mean()
    d = k.ewm(com=2, adjust=False).mean()
    j = 3 * k - 2 * d
    return k, d, j

k, d, j = calc_kdj(df["high"], df["low"], df["close"])
df["KDJ_K"] = k
df["KDJ_D"] = d
df["KDJ_J"] = j
print("KDJ计算完成")
df[["trade_date", "close", "KDJ_K", "KDJ_D", "KDJ_J"]].tail(10)

## 七、ATR（平均真实波幅）

### 7.1 ATR的计算方法

ATR（Average True Range）由J. Welles Wilder提出，用于衡量市场波动的剧烈程度，是风险管理和仓位配置的核心指标。

**计算步骤（经典参数14）：**

1. 计算真实波幅TR（取以下三者最大值）：
   - TR1 = high - low（当日波幅）
   - TR2 = |high - close[t-1]|（当前最高价与前收盘之差）
   - TR3 = |low - close[t-1]|（当前最低价与前收盘之差）
2. 计算ATR：`ATR = SMA(TR, 14)`

### 7.2 ATR的作用

- **波动率衡量**：ATR越大波动越剧烈，ATR越小市场越平静
- **动态止损**：止损价 = 当前价 - N × ATR（N通常取2或3）
- **仓位管理**：ATR大时减仓，ATR小时加仓
- **趋势确认**：ATR放大通常伴随趋势行情

In [ ]:
def calc_atr(high, low, close, period=14):
    """计算ATR指标"""
    tr1 = high - low
    tr2 = abs(high - close.shift(1))
    tr3 = abs(low - close.shift(1))
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    return tr.rolling(window=period).mean()

df["ATR_14"] = calc_atr(df["high"], df["low"], df["close"])
print("ATR计算完成")
print(f"ATR最新值: {df['ATR_14'].iloc[-1]:.2f}")
df[["trade_date", "close", "ATR_14"]].tail(10)

## 八、OBV（能量潮）

### 8.1 OBV的计算方法

OBV（On-Balance Volume）由Joe Granville提出，核心理念是"量在价先"，通过将成交量与价格变化方向关联，衡量资金流入流出的累积效应。

**计算规则：**

- 若今日收盘价 > 昨日收盘价：`OBV[t] = OBV[t-1] + vol[t]`
- 若今日收盘价 < 昨日收盘价：`OBV[t] = OBV[t-1] - vol[t]`
- 若今日收盘价 = 昨日收盘价：`OBV[t] = OBV[t-1]`

### 8.2 OBV的作用

- **量价验证**：OBV与价格同向 → 趋势健康；OBV与价格背离 → 趋势可能反转
- **底背离**：价格新低但OBV未新低 → 可能筑底（买入信号）
- **顶背离**：价格新高但OBV未新高 → 可能见顶（卖出信号）

In [ ]:
def calc_obv(close, vol):
    """计算OBV指标"""
    obv = [0]
    for i in range(1, len(close)):
        if close.iloc[i] > close.iloc[i-1]:
            obv.append(obv[-1] + vol.iloc[i])
        elif close.iloc[i] < close.iloc[i-1]:
            obv.append(obv[-1] - vol.iloc[i])
        else:
            obv.append(obv[-1])
    return pd.Series(obv, index=close.index)

df["OBV"] = calc_obv(df["close"], df["vol"])
print("OBV计算完成")
df[["trade_date", "close", "vol", "OBV"]].tail(10)

## 九、技术指标可视化

In [ ]:
# 图1: RSI指标
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df["trade_date"], df["RSI_14"], label="RSI(14)", color="#1f77b4", linewidth=1.5)
ax.axhline(y=80, color="red", linestyle="--", linewidth=1, alpha=0.7, label="超买线(80)")
ax.axhline(y=20, color="green", linestyle="--", linewidth=1, alpha=0.7, label="超卖线(20)")
ax.axhline(y=50, color="gray", linestyle=":", linewidth=0.8, alpha=0.5)
ax.fill_between(df["trade_date"], 80, 100, alpha=0.1, color="red")
ax.fill_between(df["trade_date"], 0, 20, alpha=0.1, color="green")
ax.set_title("芯动联科 RSI(14)指标", fontsize=13)
ax.set_ylabel("RSI")
ax.set_xlabel("交易日期")
ax.set_ylim(0, 100)
ax.legend()
ax.grid(True, linestyle=":", alpha=0.5)
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig(r"C:\Users\TaoZi\Desktop\ED\BA工作坊\TASK2\TASK2_ipynb_RSI.png", dpi=150, bbox_inches="tight")
print("RSI图已保存")

In [ ]:
# 图2: MACD指标
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True,
                               gridspec_kw={"height_ratios": [2, 1]})
ax1.plot(df["trade_date"], df["close"], color="#1f77b4", linewidth=1.2, label="收盘价")
ax1.plot(df["trade_date"], df["close"].rolling(5).mean(), label="MA5", color="#ff7f0e", linewidth=1)
ax1.plot(df["trade_date"], df["close"].rolling(20).mean(), label="MA20", color="#d62728", linewidth=1)
ax1.set_title("MACD指标分析", fontsize=13)
ax1.set_ylabel("价格(元)")
ax1.legend()
ax1.grid(True, linestyle=":", alpha=0.5)
colors = ["#d62728" if v >= 0 else "#2ca02c" for v in df["MACD_hist"]]
ax2.plot(df["trade_date"], df["MACD"], label="MACD线", color="#1f77b4", linewidth=1.2)
ax2.plot(df["trade_date"], df["MACD_signal"], label="信号线", color="#ff7f0e", linewidth=1.2)
ax2.bar(df["trade_date"], df["MACD_hist"], color=colors, alpha=0.7, width=0.8)
ax2.axhline(y=0, color="black", linestyle="-", linewidth=0.8)
ax2.set_ylabel("MACD")
ax2.set_xlabel("交易日期")
ax2.legend()
ax2.grid(True, linestyle=":", alpha=0.5)
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig(r"C:\Users\TaoZi\Desktop\ED\BA工作坊\TASK2\TASK2_ipynb_MACD.png", dpi=150, bbox_inches="tight")
print("MACD图已保存")

In [ ]:
# 图3: 布林带指标
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(df["trade_date"], df["close"], label="收盘价", color="#1f77b4", linewidth=1.5)
ax.plot(df["trade_date"], df["BB_upper"], label="布林上轨", color="#ff7f0e", linestyle="--", linewidth=1)
ax.plot(df["trade_date"], df["BB_mid"], label="布林中轨", color="#2ca02c", linestyle="--", linewidth=1)
ax.plot(df["trade_date"], df["BB_lower"], label="布林下轨", color="#ff7f0e", linestyle="--", linewidth=1)
ax.fill_between(df["trade_date"], df["BB_upper"], df["BB_lower"], alpha=0.1, color="orange")
ax.set_title("布林带(Bollinger Bands)指标", fontsize=13)
ax.set_ylabel("价格(元)")
ax.set_xlabel("交易日期")
ax.legend()
ax.grid(True, linestyle=":", alpha=0.5)
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig(r"C:\Users\TaoZi\Desktop\ED\BA工作坊\TASK2\TASK2_ipynb_BB.png", dpi=150, bbox_inches="tight")
print("布林带图已保存")

## 十、保存完整技术指标数据

将所有计算得到的指标保存到CSV文件，以备后续任务使用。

In [ ]:
output_path = r"C:\Users\TaoZi\Desktop\ED\BA工作坊\TASK2\TASK2_完整技术指标数据.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"完整指标数据已保存: {output_path}")
print(f"共 {len(df)} 条记录, {len(df.columns)} 个字段")
print("\n字段列表:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i}. {col}")

## 十一、指标汇总与小结

### 各指标对比

| 指标 | 类型 | 主要用途 | 经典参数 |
|------|------|----------|----------|
| RSI | 摆动指标 | 超买超卖、背离 | 14日 |
| MACD | 趋势指标 | 趋势判断、买卖信号 | (12,26,9) |
| 布林带 | 波动指标 | 超买超卖、波动率 | (20,2) |
| KDJ | 摆动指标 | 超买超卖、买卖信号 | 9日 |
| ATR | 波动指标 | 风险管理、止损 | 14日 |
| OBV | 量能指标 | 量价关系、背离 | 累积计算 |

### 使用建议

1. **多指标综合验证**：单一指标可能出现虚假信号，建议至少2-3个指标共同验证后再做决策
2. **结合价格行为**：技术指标应结合K线形态、支撑阻力位等进行综合判断
3. **风险管理优先**：使用ATR设置动态止损，控制每笔交易的风险敞口
4. **注意指标局限性**：所有技术指标均基于历史数据，存在滞后性，不构成投资建议